<a id="softmax"></a>

$$
\text{Softmax 回归 (Softmax Regression / Multinomial Logistic Regression): K 分类} \\ \quad \\
\text{每个类别一个线性打分 (logit): } z_k = w_k^T x, \ k = 1, \cdots, K \text{, 写成 } z = W^T x \ (W \in \mathbb{R}^{d \times K}) \\
\text{Softmax 把打分变成概率 (全部为正, 和为 1):} \\
p_k = \text{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}} \\ \quad \\
\text{推导动机: 让每一类相对于参考类 K 的对数几率是线性的 } \ln \frac{p_k}{p_K} = z_k - z_K \\
\implies p_k = p_K e^{z_k - z_K} \text{, 再由 } \sum_k p_k = 1 \text{ 得 } p_K = \frac{e^{z_K}}{\sum_j e^{z_j}} \text{, 代回即为 softmax} \\ \quad \\
\text{K = 2 时退化成 sigmoid: } p_1 = \frac{e^{z_1}}{e^{z_0} + e^{z_1}} = \frac{1}{1 + e^{-(z_1 - z_0)}} = \sigma(z_1 - z_0) \\ \quad \\
\text{平移不变性: } \text{softmax}(z + c \mathbf{1}) = \frac{e^{z_k} e^{c}}{\sum_j e^{z_j} e^{c}} = \text{softmax}(z) \\
\text{所以参数不唯一 (每一列 } w_k \text{ 同时加上同一个向量不改变概率), 实际中用正则化或约束 } \textstyle\sum_k w_k = 0
$$

数学基础: 二分类的情况: [逻辑回归](logistic_regression.ipynb#logistic-model) · [Sigmoid 与对数几率](sigmoid_function.ipynb#log-odds)

<a id="softmax-stability"></a>

$$
\text{数值稳定性 (Numerical Stability): 减去最大值} \\
z = (1000, 1001, 1002) \text{ 时 } e^{1000} \text{ 超出 float64 范围 (约 } 1.8 \times 10^{308} \text{), 直接算得到 inf / inf = nan} \\
\text{利用平移不变性, 令 } m = \max_j z_j \text{:} \\
p_k = \frac{e^{z_k - m}}{\sum_j e^{z_j - m}} \quad (\text{所有指数 } \leq 0 \text{, 不会溢出, 分母至少为 1}) \\ \quad \\
\text{计算交叉熵时需要 } \ln p_k \text{, 用对数求和指数 (log-sum-exp):} \\
\ln \sum_j e^{z_j} = \ln \left( e^m \sum_j e^{z_j - m} \right) = m + \ln \sum_j e^{z_j - m} \\
\ln p_k = z_k - \ln \sum_j e^{z_j} = z_k - m - \ln \sum_j e^{z_j - m} \quad (\text{避免 } p_k \text{ 下溢成 0 后再取 ln})
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax as scipy_softmax, logsumexp
from sklearn.linear_model import LogisticRegression

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def softmax(z):
    z_shift = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z_shift)
    return e / e.sum(axis=-1, keepdims=True)

def log_softmax(z):
    m = z.max(axis=-1, keepdims=True)
    return z - m - np.log(np.exp(z - m).sum(axis=-1, keepdims=True))

z_small = np.array([1.0, 2.0, 3.0])
z_big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over='ignore', invalid='ignore'):
    print(f'z = {z_big}: 直接算 = {softmax_naive(z_big)}, 减去最大值 = {softmax(z_big)}')
print(f'平移不变: softmax(z) == softmax(z + 1000) ? {np.allclose(softmax(z_small), softmax(z_big))}, == scipy? {np.allclose(softmax(z_big), scipy_softmax(z_big))}')
print(f'log-sum-exp: 手算 = {z_big.max() + np.log(np.exp(z_big - z_big.max()).sum()):.6f}, scipy = {logsumexp(z_big):.6f}')
z_extreme = np.array([0.0, 800.0])
with np.errstate(divide='ignore'):
    print(f'z = {z_extreme}: ln(softmax) 直接算 = {np.log(softmax(z_extreme))}, log-softmax = {log_softmax(z_extreme)}')

# K = 2 退化成 sigmoid
z2 = np.array([0.3, 1.7])
print(f'K = 2: softmax 第二个分量 = {softmax(z2)[1]:.6f}, sigmoid(z1 - z0) = {1 / (1 + np.exp(-(z2[1] - z2[0]))):.6f}')

<a id="softmax-jacobian"></a>

$$
\text{Softmax 的 Jacobian: } \frac{\partial p_i}{\partial z_j} \text{, 用商法则, 记 } S = \sum_k e^{z_k} \text{, } \frac{\partial S}{\partial z_j} = e^{z_j} \\ \quad \\
\text{1. } i = j \text{: } \frac{\partial}{\partial z_i} \frac{e^{z_i}}{S} = \frac{e^{z_i} S - e^{z_i} e^{z_i}}{S^2} = \frac{e^{z_i}}{S} \left(1 - \frac{e^{z_i}}{S}\right) = p_i (1 - p_i) \\
\text{2. } i \neq j \text{: 分子 } e^{z_i} \text{ 不含 } z_j \text{, } \frac{\partial}{\partial z_j} \frac{e^{z_i}}{S} = \frac{0 \cdot S - e^{z_i} e^{z_j}}{S^2} = -p_i p_j \\ \quad \\
\text{合并 (用 Kronecker delta } \delta_{ij} \text{): } \frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j) \\
J = \text{diag}(p) - p p^T \\ \quad \\
\text{性质: 每一行求和 } \sum_j p_i (\delta_{ij} - p_j) = p_i - p_i \sum_j p_j = 0 \\
\text{(所有 z 同时增加相同的量, 概率不变: } J \mathbf{1} = 0 \text{, 这就是平移不变性的微分形式)} \\
\text{J 对称且半正定: } v^T J v = \sum_i p_i v_i^2 - (\sum_i p_i v_i)^2 = Var_p(v) \geq 0
$$

数学基础: [商法则](../math_basics/calculus.ipynb#derivative-rules) · [Jacobian 矩阵](../math_basics/calculus.ipynb#jacobian) · [方差 = E(X^2) - E(X)^2](../math_basics/statistics.ipynb#expectation-variance-properties)

In [ ]:
def numerical_jacobian(F, x, h=1e-6):
    cols = [(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h]
    return np.array(cols).T

z0 = np.array([0.5, -1.0, 2.0, 0.1])
p0 = softmax(z0)
J_formula = np.diag(p0) - np.outer(p0, p0)
J_num = numerical_jacobian(softmax, z0)
print(f'Jacobian 公式 == 数值? {np.allclose(J_formula, J_num, atol=1e-8)}')
print(f'每行之和的最大绝对值 = {np.abs(J_formula.sum(axis=1)).max():.1e}, 对称? {np.allclose(J_formula, J_formula.T)}, 特征值 = {np.round(np.linalg.eigvalsh(J_formula), 6) + 0.0} (>= 0, 有一个 0)')

<a id="softmax-gradient"></a>

$$
\text{交叉熵损失与梯度} \\ \quad \\
\text{标签用独热编码 (one-hot) } y \in \{0, 1\}^K \text{, 真实类别为 c 时 } y_c = 1 \\
\text{类别分布 (Categorical Distribution) 的似然: } P(y \mid x) = \prod_k p_k^{y_k} = p_c \quad (\text{伯努利分布的推广}) \\
\text{负对数似然 = 交叉熵: } \ell = -\sum_{k=1}^K y_k \ln p_k = -\ln p_c \\ \quad \\
\text{方法 1 (链式法则 + Jacobian):} \\
\frac{\partial \ell}{\partial z_j} = \sum_k \frac{\partial \ell}{\partial p_k} \frac{\partial p_k}{\partial z_j} = \sum_k \left(-\frac{y_k}{p_k}\right) p_k (\delta_{kj} - p_j) = -\sum_k y_k \delta_{kj} + p_j \sum_k y_k = -y_j + p_j \\ \quad \\
\text{方法 2 (先化简): } \ell = -\ln \frac{e^{z_c}}{\sum_j e^{z_j}} = -z_c + \ln \sum_j e^{z_j} \implies \frac{\partial \ell}{\partial z_j} = -\delta_{cj} + \frac{e^{z_j}}{\sum_i e^{z_i}} = p_j - y_j \\ \quad \\
\nabla_z \ell = p - y \quad (\text{和逻辑回归的 } p - y \text{ 一样}) \\ \quad \\
\text{对参数: } z_k = w_k^T x \implies \frac{\partial \ell}{\partial w_k} = (p_k - y_k) x \implies \frac{\partial \ell}{\partial W} = x (p - y)^T \ (d \times K) \\
\text{n 个样本 (P, Y 是 n x K 矩阵): } J(W) = -\frac{1}{n} \sum_i \sum_k Y_{ik} \ln P_{ik}, \quad \nabla_W J = \frac{1}{n} X^T (P - Y) \\ \quad \\
\text{从 0 开始梯度下降时: } (P - Y) \text{ 每一行求和为 } 1 - 1 = 0 \text{, 所以梯度的各列之和为 0, W 的各列之和始终为 0 (自动满足约束)}
$$

数学基础: [链式法则](../math_basics/calculus.ipynb#derivative-rules) · [最大似然估计](../math_basics/statistics.ipynb#mle) · [伯努利分布的 MLE](../math_basics/statistics.ipynb#mle-bernoulli)

In [ ]:
def cross_entropy(W, X, Y):
    return -np.mean(np.sum(Y * log_softmax(X @ W), axis=1))

def ce_grad(W, X, Y):
    return X.T @ (softmax(X @ W) - Y) / len(X)

# 单个样本: dl/dz = p - y, 两种方法 + 数值验证
y_one = np.array([0, 0, 1, 0])
dl_dz_chain = J_formula.T @ (-y_one / p0)
loss_z = lambda z: -np.sum(y_one * log_softmax(z))
print(f'dl/dz: 链式法则 = {dl_dz_chain}, p - y = {p0 - y_one}, 数值 = {numerical_jacobian(lambda z: np.array([loss_z(z)]), z0).ravel()}')

# 3 类数据
rng = np.random.default_rng(1)
centers = np.array([[0.0, 2.0], [-2.0, -1.0], [2.0, -1.0]])
n_per = 120
X_raw = np.vstack([rng.normal(c, 1.1, size=(n_per, 2)) for c in centers])
labels = np.repeat(np.arange(3), n_per)
Y = np.eye(3)[labels]                                # one-hot
X = np.column_stack([np.ones(len(X_raw)), X_raw])

W_test = rng.normal(size=(3, 3))
num_grad = np.zeros_like(W_test)
for idx in np.ndindex(W_test.shape):
    E = np.zeros_like(W_test); E[idx] = 1e-6
    num_grad[idx] = (cross_entropy(W_test + E, X, Y) - cross_entropy(W_test - E, X, Y)) / 2e-6
print(f'梯度 X^T (P - Y) / n == 数值? {np.allclose(ce_grad(W_test, X, Y), num_grad, atol=1e-8)}')

<a id="softmax-training"></a>

$$
\text{训练: 梯度下降 } W \leftarrow W - \eta \frac{1}{n} X^T (P - Y) \\
\text{预测: } \hat{y} = \arg\max_k p_k = \arg\max_k z_k \text{ (softmax 单调, 不改变大小顺序)} \\
\text{决策边界: 类别 i 和 j 的边界 } p_i = p_j \iff z_i = z_j \iff (w_i - w_j)^T x = 0 \text{, 仍然是直线}
$$

In [ ]:
W = np.zeros((3, 3))
losses = []
eta = 0.5
for it in range(20000):
    W = W - eta * ce_grad(W, X, Y)
    if it % 100 == 0:
        losses.append(cross_entropy(W, X, Y))
print(f'梯度下降后 |grad| = {np.linalg.norm(ce_grad(W, X, Y)):.2e}, W 每行之和 (各类别参数之和) 的最大绝对值 = {np.abs(W.sum(axis=1)).max():.1e}')

sk = LogisticRegression(C=1e12, tol=1e-12, max_iter=20000).fit(X_raw, labels)
W_sk = np.vstack([sk.intercept_, sk.coef_.T])         # sklearn 的参数, 同样满足各类之和为 0
print(f'手算 W = \n{W}\nsklearn W = \n{W_sk}')
print(f'参数一致? {np.allclose(W, W_sk, atol=1e-3)}, predict_proba 一致? {np.allclose(softmax(X @ W), sk.predict_proba(X_raw), atol=1e-5)}')
pred = np.argmax(X @ W, axis=1)
print(f'准确率: 手算 = {np.mean(pred == labels):.4f}, sklearn = {sk.score(X_raw, labels):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(np.arange(len(losses)) * 100, losses, label='cross-entropy')
axs[0].set_xlabel('iteration')
g1, g2 = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-5, 5, 300))
grid = np.column_stack([np.ones(g1.size), g1.ravel(), g2.ravel()])
region = np.argmax(grid @ W, axis=1).reshape(g1.shape)
axs[1].contourf(g1, g2, region, levels=[-0.5, 0.5, 1.5, 2.5], colors=['C0', 'C1', 'C2'], alpha=0.25)
for k in range(3):
    axs[1].scatter(*X_raw[labels == k].T, s=6, color=f'C{k}', label=f'class {k}')
plot_nicely()
plt.show()